# 03 · Feature engineering

Build the model table **without leakage**: every feature at hour *t* uses only data available at *t*.

- Lags of observed PM2.5 (`shift(+k)`)
- Hotspot counts / upwind exposure from detections with `acquired_at ≤ t`
- Target = observed PM2.5 at `t + h` (`shift(-h)`)

Shown here on the synthetic fixture to demonstrate the pipeline only.

In [ ]:
from pavana_ml.data.fixtures import load_scenario, scenario_frames
from pavana_ml.features.lags import add_lags, add_target
from pavana_ml.features.upwind import UpwindParams, upwind_series

scenario = load_scenario()
hourly, hotspots = scenario_frames(scenario)
loc = scenario["location"]

up = upwind_series(hourly, hotspots, loc["lat"], loc["lon"], window_hours=48, params=UpwindParams())
table = hourly.merge(up, on="valid_at")
table = add_target(add_lags(table, "pm25_ug_m3", (1, 3, 6)), "pm25_ug_m3", horizon_hours=6)
table.head(10)

**Interpretation guardrail:** `upwind_score` is an indicator combining FRP, distance decay,
wind alignment and recency. It is *not* a probability and *not* a plume model.